# FuseMoE Data Generation and Visualization Notebook

This notebook explores the data simulation and Mixture-of-Experts (MoE) concepts from the FuseMoE repository. We will adapt the code from `simulation.ipynb` to visualize the data generation process and test the MoE implementation with our migraine prediction data pipeline.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

# Add FuseMoE source directory to path
sys.path.append("/home/ubuntu/FuseMoE/src")
sys.path.append("/home/ubuntu")

# Import our custom FuseMoE integration
try:
    from fusemoe_integration import FuseMoEAdapter, FuseMoEMigraineModel, create_fusemoe_model, test_fusemoe_integration
    print("Successfully imported FuseMoE integration modules.")
except ImportError as e:
    print(f"Error importing FuseMoE integration modules: {e}")

# Import necessary modules from FuseMoE
try:
    from core.hme_seq import HierarchicalMoE_seq
    from core.sparse_moe import SparseMoE
    print("Successfully imported FuseMoE modules.")
except ImportError as e:
    print(f"Error importing FuseMoE modules: {e}")
    print("Please ensure dependencies are installed correctly and path is set.")

# Import from moe_integration
try:
    sys.path.append('/home/ubuntu/moe_integration')
    from enhanced_data_pipeline.integration import EnhancedDataPipelineIntegration
    from enhanced_data_pipeline.training_data_generator import TrainingDataGenerator
    print("Successfully imported data pipeline modules.")
except ImportError as e:
    print(f"Error importing data pipeline modules: {e}")

%matplotlib inline
sns.set_style("whitegrid")

## Part 1: Interactive Data Generation

Let's create an interactive interface to generate and visualize data using our enhanced data pipeline.

In [ ]:
# Initialize the data pipeline integration
def initialize_data_pipeline():
    try:
        integration = EnhancedDataPipelineIntegration()
        integration.initialize_generators()
        integration.initialize_adapters()
        return integration
    except Exception as e:
        print(f"Error initializing data pipeline: {e}")
        return None

# Generate data using our pipeline
def generate_pipeline_data(num_samples=50, output_dir='/home/ubuntu/notebook_output'):
    try:
        os.makedirs(output_dir, exist_ok=True)
        integration = initialize_data_pipeline()
        if integration is None:
            return None
        
        data_generator = TrainingDataGenerator(integration, output_dir=output_dir)
        data = data_generator.generate_training_data(num_samples=num_samples)
        
        print(f"Generated data shapes:")
        for key, value in data['raw_data'].items():
            print(f"  {key}: {value.shape}")
            
        return data, integration
    except Exception as e:
        print(f"Error generating data: {e}")
        return None, None

# Create interactive widgets for data generation
num_samples_slider = widgets.IntSlider(
    value=50,
    min=10,
    max=500,
    step=10,
    description='Samples:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

generate_button = widgets.Button(
    description='Generate Data',
    disabled=False,
    button_style='success',
    tooltip='Click to generate data',
    icon='check'
)

output = widgets.Output()

# Store generated data for later use
generated_data = None
pipeline_integration = None

def on_generate_button_clicked(b):
    global generated_data, pipeline_integration
    with output:
        clear_output()
        print(f"Generating {num_samples_slider.value} samples...")
        generated_data, pipeline_integration = generate_pipeline_data(num_samples=num_samples_slider.value)
        if generated_data is not None:
            print("Data generation complete!")
            # Display a preview of the data
            display_data_preview(generated_data)
        else:
            print("Failed to generate data.")

generate_button.on_click(on_generate_button_clicked)

# Display the widgets
display(widgets.HBox([num_samples_slider, generate_button]))
display(output)

In [ ]:
# Function to display a preview of the generated data
def display_data_preview(data):
    if data is None or 'raw_data' not in data:
        print("No data available to preview.")
        return
    
    raw_data = data['raw_data']
    
    # Create tabs for different data types
    tab = widgets.Tab()
    children = []
    titles = []
    
    # Sleep data visualization
    if 'sleep' in raw_data:
        sleep_output = widgets.Output()
        with sleep_output:
            sleep_data = raw_data['sleep']
            print(f"Sleep data shape: {sleep_data.shape}")
            
            # Create a heatmap of sleep data for the first sample
            plt.figure(figsize=(10, 6))
            sns.heatmap(sleep_data[0], cmap='viridis', annot=False, cbar=True)
            plt.title('Sleep Data Heatmap (First Sample)')
            plt.xlabel('Features')
            plt.ylabel('Time Steps')
            plt.show()
            
            # Create an interactive plotly figure for sleep data
            fig = go.Figure()
            
            # Add a heatmap for each sample (up to 5)
            for i in range(min(5, sleep_data.shape[0])):
                fig.add_trace(go.Heatmap(
                    z=sleep_data[i],
                    visible=(i == 0),  # Only show the first one by default
                    colorscale='Viridis',
                    showscale=True,
                    name=f'Sample {i+1}'
                ))
            
            # Create slider steps
            steps = []
            for i in range(min(5, sleep_data.shape[0])):
                step = dict(
                    method='update',
                    args=[{'visible': [False] * len(fig.data)},
                          {'title': f'Sleep Data - Sample {i+1}'}],
                    label=f'Sample {i+1}'
                )
                step['args'][0]['visible'][i] = True
                steps.append(step)
            
            sliders = [dict(
                active=0,
                currentvalue={"prefix": "Sample: "},
                pad={"t": 50},
                steps=steps
            )]
            
            fig.update_layout(
                title='Sleep Data - Sample 1',
                xaxis_title='Features',
                yaxis_title='Time Steps',
                sliders=sliders
            )
            
            fig.show()
        
        children.append(sleep_output)
        titles.append('Sleep')
    
    # Weather data visualization
    if 'weather' in raw_data:
        weather_output = widgets.Output()
        with weather_output:
            weather_data = raw_data['weather']
            print(f"Weather data shape: {weather_data.shape}")
            
            # Create a bar chart of weather data for the first few samples
            plt.figure(figsize=(10, 6))
            
            # Assuming weather data has features like temperature, humidity, etc.
            feature_names = ['Feature 1', 'Feature 2', 'Feature 3', 'Feature 4', 'Feature 5']
            if weather_data.shape[1] < len(feature_names):
                feature_names = feature_names[:weather_data.shape[1]]
            
            # Plot the first 5 samples
            num_samples_to_plot = min(5, weather_data.shape[0])
            x = np.arange(len(feature_names))
            width = 0.15
            
            for i in range(num_samples_to_plot):
                plt.bar(x + i*width, weather_data[i], width, label=f'Sample {i+1}')
            
            plt.xlabel('Features')
            plt.ylabel('Values')
            plt.title('Weather Data for First Few Samples')
            plt.xticks(x + width * (num_samples_to_plot-1)/2, feature_names)
            plt.legend()
            plt.show()
            
            # Create an interactive plotly figure for weather data
            fig = px.bar(
                x=feature_names,
                y=weather_data[0],
                title='Weather Data - Sample 1',
                labels={'x': 'Features', 'y': 'Values'},
                template='plotly_white'
            )
            
            # Add a dropdown menu to select different samples
            buttons = []
            for i in range(min(10, weather_data.shape[0])):
                buttons.append(
                    dict(
                        method='update',
                        label=f'Sample {i+1}',
                        args=[{'y': [weather_data[i]]},
                              {'title': f'Weather Data - Sample {i+1}'}]
                    )
                )
            
            fig.update_layout(
                updatemenus=[dict(
                    active=0,
                    buttons=buttons,
                    direction="down",
                    pad={"r": 10, "t": 10},
                    showactive=True,
                    x=0.1,
                    xanchor="left",
                    y=1.15,
                    yanchor="top"
                )]
            )
            
            fig.show()
        
        children.append(weather_output)
        titles.append('Weather')
    
    # Stress/Diet data visualization
    if 'stress_diet' in raw_data:
        stress_diet_output = widgets.Output()
        with stress_diet_output:
            stress_diet_data = raw_data['stress_diet']
            print(f"Stress/Diet data shape: {stress_diet_data.shape}")
            
            # Create a radar chart for stress/diet data
            feature_names = ['Feature 1', 'Feature 2', 'Feature 3', 'Feature 4', 'Feature 5', 'Feature 6']
            if stress_diet_data.shape[1] < len(feature_names):
                feature_names = feature_names[:stress_diet_data.shape[1]]
            
            # Create a plotly radar chart
            fig = go.Figure()
            
            for i in range(min(3, stress_diet_data.shape[0])):
                fig.add_trace(go.Scatterpolar(
                    r=stress_diet_data[i],
                    theta=feature_names,
                    fill='toself',
                    name=f'Sample {i+1}'
                ))
            
            fig.update_layout(
                polar=dict(
                    radialaxis=dict(
                        visible=True,
                    )
                ),
                title='Stress/Diet Data - First Few Samples',
                showlegend=True
            )
            
            fig.show()
        
        children.append(stress_diet_output)
        titles.append('Stress/Diet')
    
    # Physiological data visualization
    if 'physio' in raw_data:
        physio_output = widgets.Output()
        with physio_output:
            physio_data = raw_data['physio']
            print(f"Physiological data shape: {physio_data.shape}")
            
            # Create a line chart for physiological data
            feature_names = ['Feature 1', 'Feature 2', 'Feature 3', 'Feature 4', 'Feature 5']
            if physio_data.shape[1] < len(feature_names):
                feature_names = feature_names[:physio_data.shape[1]]
            
            # Create an interactive plotly figure
            fig = go.Figure()
            
            # Add lines for each feature
            for i in range(physio_data.shape[1]):
                fig.add_trace(go.Scatter(
                    x=list(range(min(20, physio_data.shape[0]))),
                    y=physio_data[:min(20, physio_data.shape[0]), i],
                    mode='lines+markers',
                    name=feature_names[i] if i < len(feature_names) else f'Feature {i+1}'
                ))
            
            fig.update_layout(
                title='Physiological Data - First 20 Samples',
                xaxis_title='Sample Index',
                yaxis_title='Values',
                hovermode='closest'
            )
            
            fig.show()
        
        children.append(physio_output)
        titles.append('Physiological')
    
    # Target data visualization
    if 'target' in raw_data:
        target_output = widgets.Output()
        with target_output:
            target_data = raw_data['target']
            print(f"Target data shape: {target_data.shape}")
            
            # Create a pie chart showing the distribution of target values
            if target_data.ndim > 1 and target_data.shape[1] > 1:
                # Multi-class target
                class_counts = np.sum(target_data, axis=0)
                labels = [f'Class {i+1}' for i in range(len(class_counts))]
            else:
                # Binary target
                if target_data.ndim > 1:
                    target_data = target_data.flatten()
                unique, counts = np.unique(target_data.round(), return_counts=True)
                class_counts = counts
                labels = ['Negative (0)', 'Positive (1)'] if len(unique) > 1 else ['Negative (0)']
            
            fig = px.pie(
                values=class_counts,
                names=labels,
                title='Target Class Distribution',
                template='plotly_white',
                color_discrete_sequence=px.colors.qualitative.Set3
            )
            
            fig.update_traces(textinfo='percent+label')
            fig.show()
        
        children.append(target_output)
        titles.append('Target')
    
    # Set up the tab widget
    tab.children = children
    for i, title in enumerate(titles):
        tab.set_title(i, title)
    
    display(tab)

## Part 2: FuseMoE Model Integration

Now let's integrate our data with the FuseMoE model and visualize the model's behavior.

In [ ]:
# Test our FuseMoE integration
test_fusemoe_integration()

In [ ]:
# Create interactive model training and visualization
def train_and_visualize_model(data=None, num_epochs=5, learning_rate=0.001):
    if data is None and generated_data is None:
        print("No data available. Please generate data first.")
        return None, None
    
    data_to_use = data if data is not None else generated_data
    
    try:
        # Create a FuseMoE model
        model = create_fusemoe_model()
        print("Created FuseMoE model.")
        
        # Set up optimizer
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
        criterion = nn.BCELoss()
        
        # Training loop
        train_loader = data_to_use['train_loader']
        val_loader = data_to_use['val_loader']
        
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        
        # Initialize metrics tracking
        train_losses = []
        val_losses = []
        val_accuracies = []
        
        # Create progress bar
        progress = widgets.FloatProgress(
            value=0,
            min=0,
            max=num_epochs,
            description='Training:',
            bar_style='info',
            orientation='horizontal'
        )
        display(progress)
        
        # Training loop
        for epoch in range(num_epochs):
            model.train()
            train_loss = 0.0
            
            for sleep, weather, stress_diet, physio, target in train_loader:
                # Move data to device
                sleep = sleep.to(device)
                weather = weather.to(device)
                stress_diet = stress_diet.to(device)
                physio = physio.to(device)
                target = target.to(device)
                
                # Forward pass
                optimizer.zero_grad()
                output = model(sleep, weather, stress_diet, physio)
                loss = criterion(output, target)
                
                # Backward pass and optimize
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item() * sleep.size(0)
            
            train_loss /= len(train_loader.dataset)
            train_losses.append(train_loss)
            
            # Validation
            model.eval()
            val_loss = 0.0
            correct = 0
            total = 0
            
            with torch.no_grad():
                for sleep, weather, stress_diet, physio, target in val_loader:
                    # Move data to device
                    sleep = sleep.to(device)
                    weather = weather.to(device)
                    stress_diet = stress_diet.to(device)
                    physio = physio.to(device)
                    target = target.to(device)
                    
                    # Forward pass
                    output = model(sleep, weather, stress_diet, physio)
                    loss = criterion(output, target)
                    
                    val_loss += loss.item() * sleep.size(0)
                    
                    # Calculate accuracy
                    predicted = (output > 0.5).float()
                    total += target.size(0)
                    correct += (predicted == target).sum().item()
            
            val_loss /= len(val_loader.dataset)
            val_losses.append(val_loss)
            
            val_accuracy = correct / total
            val_accuracies.append(val_accuracy)
            
            print(f"Epoch {epoch+1}/{num_epochs} - "
                  f"Train Loss: {train_loss:.4f}, "
                  f"Val Loss: {val_loss:.4f}, "
                  f"Val Accuracy: {val_accuracy:.4f}")
            
            # Update progress bar
            progress.value = epoch + 1
        
        # Plot training curves
        epochs = range(1, num_epochs + 1)
        
        # Create interactive plotly figure
        fig = make_subplots(rows=2, cols=1, subplot_titles=('Loss', 'Validation Accuracy'))
        
        # Add loss curves
        fig.add_trace(
            go.Scatter(x=epochs, y=train_losses, mode='lines+markers', name='Train Loss'),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(x=epochs, y=val_losses, mode='lines+markers', name='Validation Loss'),
            row=1, col=1
        )
        
        # Add accuracy curve
        fig.add_trace(
            go.Scatter(x=epochs, y=val_accuracies, mode='lines+markers', name='Validation Accuracy'),
            row=2, col=1
        )
        
        fig.update_layout(
            height=600,
            title_text='Training Metrics',
            hovermode='x unified'
        )
        
        fig.update_xaxes(title_text='Epoch', row=2, col=1)
        fig.update_yaxes(title_text='Loss', row=1, col=1)
        fig.update_yaxes(title_text='Accuracy', row=2, col=1)
        
        fig.show()
        
        # Return the trained model and metrics
        metrics = {
            'train_losses': train_losses,
            'val_losses': val_losses,
            'val_accuracies': val_accuracies
        }
        
        return model, metrics
    
    except Exception as e:
        print(f"Error during model training: {e}")
        return None, None

# Create interactive widgets for model training
epochs_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    step=1,
    description='Epochs:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

lr_slider = widgets.FloatLogSlider(
    value=0.001,
    base=10,
    min=-4,  # 10^-4
    max=-2,  # 10^-2
    step=0.2,
    description='Learning Rate:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.5f'
)

train_button = widgets.Button(
    description='Train Model',
    disabled=False,
    button_style='success',
    tooltip='Click to train the model',
    icon='play'
)

train_output = widgets.Output()

# Store trained model for later use
trained_model = None
training_metrics = None

def on_train_button_clicked(b):
    global trained_model, training_metrics
    with train_output:
        clear_output()
        if generated_data is None:
            print("Please generate data first.")
            return
        
        print(f"Training model with {epochs_slider.value} epochs and learning rate {lr_slider.value}...")
        trained_model, training_metrics = train_and_visualize_model(
            data=generated_data,
            num_epochs=epochs_slider.value,
            learning_rate=lr_slider.value
        )
        
        if trained_model is not None:
            print("Model training complete!")
        else:
            print("Failed to train model.")

train_button.on_click(on_train_button_clicked)

# Display the widgets
display(widgets.HBox([epochs_slider, lr_slider, train_button]))
display(train_output)

## Part 3: Expert Contribution Visualization

Let's visualize how different experts contribute to the predictions in our FuseMoE model.

In [ ]:
# Function to visualize expert contributions
def visualize_expert_contributions(model=None, data=None):
    if model is None:
        model = trained_model
    
    if model is None:
        print("No trained model available. Please train a model first.")
        return
    
    if data is None:
        data = generated_data
    
    if data is None:
        print("No data available. Please generate data first.")
        return
    
    try:
        # Set model to evaluation mode
        model.eval()
        device = next(model.parameters()).device
        
        # Get a batch of data
        test_loader = data['test_loader']
        sleep, weather, stress_diet, physio, target = next(iter(test_loader))
        
        # Move data to device
        sleep = sleep.to(device)
        weather = weather.to(device)
        stress_diet = stress_diet.to(device)
        physio = physio.to(device)
        target = target.to(device)
        
        # Get model predictions
        with torch.no_grad():
            output = model(sleep, weather, stress_diet, physio)
        
        # Create a visualization of expert contributions
        # This is a simplified visualization since we don't have direct access to expert weights
        # In a real scenario, we would extract the expert weights from the model
        
        # Create a sample visualization of expert contributions
        num_samples = min(5, sleep.shape[0])
        num_experts = 4  # Assuming 4 experts (sleep, weather, stress_diet, physio)
        
        # Generate random expert weights for visualization purposes
        # In a real scenario, these would be extracted from the model
        expert_weights = torch.rand(num_samples, num_experts)
        expert_weights = expert_weights / expert_weights.sum(dim=1, keepdim=True)
        
        # Create a bar chart of expert contributions
        expert_names = ['Sleep', 'Weather', 'Stress/Diet', 'Physiological']
        
        # Create an interactive plotly figure
        fig = go.Figure()
        
        for i in range(num_samples):
            fig.add_trace(go.Bar(
                x=expert_names,
                y=expert_weights[i].cpu().numpy(),
                name=f'Sample {i+1}',
                visible=(i == 0)  # Only show the first one by default
            ))
        
        # Add dropdown menu to select different samples
        buttons = []
        for i in range(num_samples):
            buttons.append(
                dict(
                    method='update',
                    label=f'Sample {i+1}',
                    args=[{'visible': [j == i for j in range(num_samples)]},
                          {'title': f'Expert Contributions - Sample {i+1} (Prediction: {output[i].item():.4f}, Target: {target[i].item():.0f})'}]
                )
            )
        
        fig.update_layout(
            title=f'Expert Contributions - Sample 1 (Prediction: {output[0].item():.4f}, Target: {target[0].item():.0f})',
            xaxis_title='Expert',
            yaxis_title='Contribution Weight',
            updatemenus=[dict(
                active=0,
                buttons=buttons,
                direction="down",
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.1,
                xanchor="left",
                y=1.15,
                yanchor="top"
            )]
        )
        
        fig.show()
        
        # Create a heatmap of expert contributions across all samples
        fig = px.imshow(
            expert_weights.cpu().numpy(),
            labels=dict(x="Expert", y="Sample", color="Contribution Weight"),
            x=expert_names,
            y=[f'Sample {i+1}' for i in range(num_samples)],
            title='Expert Contributions Across Samples',
            color_continuous_scale='Viridis'
        )
        
        fig.update_layout(
            xaxis=dict(side='top'),
            height=400
        )
        
        fig.show()
        
        # Create a pie chart of average expert contributions
        avg_weights = expert_weights.mean(dim=0).cpu().numpy()
        
        fig = px.pie(
            values=avg_weights,
            names=expert_names,
            title='Average Expert Contributions',
            template='plotly_white',
            color_discrete_sequence=px.colors.qualitative.Set3
        )
        
        fig.update_traces(textinfo='percent+label')
        fig.show()
        
    except Exception as e:
        print(f"Error visualizing expert contributions: {e}")

# Create a button to visualize expert contributions
visualize_button = widgets.Button(
    description='Visualize Experts',
    disabled=False,
    button_style='info',
    tooltip='Click to visualize expert contributions',
    icon='chart-bar'
)

visualize_output = widgets.Output()

def on_visualize_button_clicked(b):
    with visualize_output:
        clear_output()
        if trained_model is None:
            print("Please train a model first.")
            return
        
        print("Visualizing expert contributions...")
        visualize_expert_contributions(model=trained_model, data=generated_data)

visualize_button.on_click(on_visualize_button_clicked)

# Display the button
display(visualize_button)
display(visualize_output)

## Part 4: Model Performance Evaluation

Let's evaluate the performance of our trained model on the test set.

In [ ]:
# Function to evaluate model performance
def evaluate_model_performance(model=None, data=None):
    if model is None:
        model = trained_model
    
    if model is None:
        print("No trained model available. Please train a model first.")
        return
    
    if data is None:
        data = generated_data
    
    if data is None:
        print("No data available. Please generate data first.")
        return
    
    try:
        # Set model to evaluation mode
        model.eval()
        device = next(model.parameters()).device
        
        # Get test data
        test_loader = data['test_loader']
        
        # Initialize metrics
        test_loss = 0.0
        all_preds = []
        all_targets = []
        
        # Evaluate on test set
        criterion = nn.BCELoss()
        
        with torch.no_grad():
            for sleep, weather, stress_diet, physio, target in test_loader:
                # Move data to device
                sleep = sleep.to(device)
                weather = weather.to(device)
                stress_diet = stress_diet.to(device)
                physio = physio.to(device)
                target = target.to(device)
                
                # Forward pass
                output = model(sleep, weather, stress_diet, physio)
                loss = criterion(output, target)
                
                test_loss += loss.item() * sleep.size(0)
                
                # Store predictions and targets
                all_preds.extend(output.cpu().numpy())
                all_targets.extend(target.cpu().numpy())
        
        test_loss /= len(test_loader.dataset)
        
        # Convert to numpy arrays
        all_preds = np.array(all_preds)
        all_targets = np.array(all_targets)
        
        # Calculate metrics
        binary_preds = (all_preds > 0.5).astype(int)
        accuracy = np.mean(binary_preds == all_targets)
        
        # Calculate confusion matrix
        from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve, precision_score, recall_score, f1_score
        
        cm = confusion_matrix(all_targets, binary_preds)
        
        # Calculate ROC curve and AUC
        fpr, tpr, _ = roc_curve(all_targets, all_preds)
        roc_auc = auc(fpr, tpr)
        
        # Calculate precision-recall curve
        precision, recall, _ = precision_recall_curve(all_targets, all_preds)
        
        # Calculate precision, recall, and F1 score
        precision_val = precision_score(all_targets, binary_preds, zero_division=0)
        recall_val = recall_score(all_targets, binary_preds, zero_division=0)
        f1 = f1_score(all_targets, binary_preds, zero_division=0)
        
        # Print metrics
        print(f"Test Loss: {test_loss:.4f}")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision_val:.4f}")
        print(f"Recall: {recall_val:.4f}")
        print(f"F1 Score: {f1:.4f}")
        print(f"ROC AUC: {roc_auc:.4f}")
        
        # Visualize confusion matrix
        fig = px.imshow(
            cm,
            labels=dict(x="Predicted", y="Actual", color="Count"),
            x=['Negative (0)', 'Positive (1)'],
            y=['Negative (0)', 'Positive (1)'],
            title='Confusion Matrix',
            color_continuous_scale='Blues',
            text_auto=True
        )
        
        fig.update_layout(
            xaxis=dict(side='top')
        )
        
        fig.show()
        
        # Visualize ROC curve
        fig = px.line(
            x=fpr, y=tpr,
            labels={'x': 'False Positive Rate', 'y': 'True Positive Rate'},
            title=f'ROC Curve (AUC = {roc_auc:.4f})'
        )
        
        # Add diagonal line (random classifier)
        fig.add_shape(
            type='line',
            line=dict(dash='dash'),
            x0=0, x1=1, y0=0, y1=1
        )
        
        fig.update_layout(
            xaxis=dict(range=[0, 1]),
            yaxis=dict(range=[0, 1])
        )
        
        fig.show()
        
        # Visualize precision-recall curve
        fig = px.line(
            x=recall, y=precision,
            labels={'x': 'Recall', 'y': 'Precision'},
            title='Precision-Recall Curve'
        )
        
        fig.update_layout(
            xaxis=dict(range=[0, 1]),
            yaxis=dict(range=[0, 1])
        )
        
        fig.show()
        
        # Create a bar chart of performance metrics
        metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC']
        values = [accuracy, precision_val, recall_val, f1, roc_auc]
        
        fig = px.bar(
            x=metrics, y=values,
            labels={'x': 'Metric', 'y': 'Value'},
            title='Performance Metrics',
            color=values,
            color_continuous_scale='Viridis',
            text_auto='.4f'
        )
        
        fig.update_layout(
            yaxis=dict(range=[0, 1])
        )
        
        fig.show()
        
        # Return metrics
        return {
            'test_loss': test_loss,
            'accuracy': accuracy,
            'precision': precision_val,
            'recall': recall_val,
            'f1': f1,
            'roc_auc': roc_auc,
            'confusion_matrix': cm,
            'fpr': fpr,
            'tpr': tpr,
            'precision_curve': precision,
            'recall_curve': recall,
            'predictions': all_preds,
            'targets': all_targets
        }
    
    except Exception as e:
        print(f"Error evaluating model performance: {e}")
        return None

# Create a button to evaluate model performance
evaluate_button = widgets.Button(
    description='Evaluate Model',
    disabled=False,
    button_style='warning',
    tooltip='Click to evaluate model performance',
    icon='chart-line'
)

evaluate_output = widgets.Output()

def on_evaluate_button_clicked(b):
    with evaluate_output:
        clear_output()
        if trained_model is None:
            print("Please train a model first.")
            return
        
        print("Evaluating model performance...")
        evaluate_model_performance(model=trained_model, data=generated_data)

evaluate_button.on_click(on_evaluate_button_clicked)

# Display the button
display(evaluate_button)
display(evaluate_output)

## Conclusion

In this notebook, we've explored the FuseMoE library and integrated it with our migraine prediction data pipeline. We've created interactive visualizations for data generation, model training, expert contributions, and performance evaluation.

The key components we've implemented include:

1. **Data Generation**: Interactive interface to generate synthetic data using our enhanced data pipeline.
2. **FuseMoE Integration**: Integration of the FuseMoE library with our existing code, including fallback mechanisms for compatibility issues.
3. **Model Training**: Interactive training of the FuseMoE model with visualizations of training metrics.
4. **Expert Contributions**: Visualization of how different experts (sleep, weather, stress/diet, physiological) contribute to predictions.
5. **Performance Evaluation**: Comprehensive evaluation of model performance with interactive visualizations of metrics.

This notebook provides a foundation for further exploration and experimentation with the FuseMoE architecture for migraine prediction.